### WP1.9 - Load Allocation

This tutorial shows how to prove the proper operation of the "cap to transformer rating" functionality


In [1]:
# Import needed libraries
import multiconductor as mc
from multiconductor.load_allocation.load_allocation import run_load_allocation, build_measurement_graph
import pickle
import numpy as np

In [2]:
# Import pp-mc grid
filename = "CKT_1022_18104.pkl"  # NB: provide the correct directory of the file
with open(filename, "rb") as f:
    net = pickle.load(f)

In [3]:
# Create graph of the grid
mg = build_measurement_graph(net)

In [4]:
# Create measurement to be used for the load allocation
mc.create_measurement(net, measurement_type="p", element_type="line", element=148,
                        value=float(9.70), std_dev=0, side="from")

In [6]:
# Run load allocation without any cap
run_load_allocation(net, mg)


 run_load_allocation for measurement None
[WARN] loads exceed rating (factor=1.0): [(27, 0)]
Iter 0: measurement_target=9.7000 MW, sim=8.6450 MW, delta=10.877%
Iter 1: measurement_target=9.7000 MW, sim=9.7305 MW, delta=-0.314%
Converged within ±0.5% after 1 iterations


In [7]:
# Print loads (the first 15 in the list) as a term of comparison
net.asymmetric_load.head(15)

,,name,bus,from_phase,to_phase,p_mw,q_mvar,const_z_percent_p,const_i_percent_p,const_z_percent_q,const_i_percent_q,sn_mva,scaling,in_service,type
index,circuit,,,,,,,,,,,,,,
0,0,5169438,739,1,0,0.024887,0.005569,0.0,0.0,0.0,0.0,0.025,1.0,True,commercial
1,0,5598137,410,1,0,0.049773,0.000730,0.0,0.0,0.0,0.0,0.050,1.0,True,commercial
2,0,5169550,560,2,0,0.074660,0.011428,0.0,0.0,0.0,0.0,0.075,1.0,True,agricultural
3,0,5498950,117,3,0,0.049773,0.008638,0.0,0.0,0.0,0.0,0.050,1.0,True,residential
4,0,5377960,671,2,0,0.074660,0.008725,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
5,0,5377877,914,1,0,0.074660,0.014785,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
6,0,5313536,364,1,0,0.049773,0.008574,0.0,0.0,0.0,0.0,0.050,1.0,True,residential
7,0,5378060,917,3,0,0.074660,0.006189,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
8,0,5092423,506,2,0,0.049773,0.004595,0.0,0.0,0.0,0.0,0.050,1.0,True,residential


In [8]:
# Run load allocation with transformer cap
run_load_allocation(net, mg, cap_to_transformer_rating=True, trafo_overload_factor=1.5)


 run_load_allocation for measurement None
Iter 0: measurement_target=9.7000 MW, sim=8.6453 MW, delta=10.873%
Iter 1: measurement_target=9.7000 MW, sim=9.2291 MW, delta=4.855%
Iter 2: measurement_target=9.7000 MW, sim=9.6913 MW, delta=0.090%
Converged within ±0.5% after 2 iterations


In [10]:
# Print again loads as a term of comparison  
# --> you will have some loads capped and some other loads with larger values because they have to compensate for the power capped in the other ones.
net.asymmetric_load.head(15)

,,name,bus,from_phase,to_phase,p_mw,q_mvar,const_z_percent_p,const_i_percent_p,const_z_percent_q,const_i_percent_q,sn_mva,scaling,in_service,type
index,circuit,,,,,,,,,,,,,,
0,0,5169438,739,1,0,0.026094,0.006474,0.0,0.0,0.0,0.0,0.025,1.0,True,commercial
1,0,5598137,410,1,0,0.052188,0.000849,0.0,0.0,0.0,0.0,0.050,1.0,True,commercial
2,0,5169550,560,2,0,0.078282,0.013286,0.0,0.0,0.0,0.0,0.075,1.0,True,agricultural
3,0,5498950,117,3,0,0.052188,0.010042,0.0,0.0,0.0,0.0,0.050,1.0,True,residential
4,0,5377960,671,2,0,0.078282,0.010143,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
5,0,5377877,914,1,0,0.078282,0.017188,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
6,0,5313536,364,1,0,0.052188,0.009967,0.0,0.0,0.0,0.0,0.050,1.0,True,residential
7,0,5378060,917,3,0,0.078282,0.007195,0.0,0.0,0.0,0.0,0.075,1.0,True,residential
8,0,5092423,506,2,0,0.037500,0.000000,0.0,0.0,0.0,0.0,0.050,1.0,True,residential


You can see that with this setup most loads have a p_mw slightly larger than their sn_va. 
Some exceptions exist, for example for loads 8 and 9. 
The reason is the lower value of sn_mva for their transformer.
To find the transformer connected to a load of interest, you can use the following code.

In [11]:
# To find the transformer connected to a load of interest, you can use: 
# Define the bus to which the load is connected: 
bus_idx = 506  # This is load 8 in this example grid
net.trafo1ph.index[net.trafo1ph.index.get_level_values(1)==bus_idx]

MultiIndex([(89, 506, 0),
            (89, 506, 1)],
           names=['index', 'bus', 'circuit'])

In [ ]:
# The first index found above is the one of the upstream transformer. 
# Its data can be visualized by: 
trafo_idx = 89   # index for the example above
net.trafo1ph.loc[trafo_idx]

name  from_phase  to_phase  \
bus circuit                                                           
610 0        293588B4C4F7EE34E0637F61810A7CE3           2         3   
    1        293588B4C4F7EE34E0637F61810A7CE3           3         2   
506 0        293588B4C4F7EE34E0637F61810A7CE3           2         0   
    1        293588B4C4F7EE34E0637F61810A7CE3           3         0   

                 vn_kv  sn_mva  vk_percent  vkr_percent  pfe_kw  i0_percent  \
bus circuit                                                                   
610 0        12.000000   0.025         0.5     0.353554     0.0    0.000001   
    1        12.000000   0.025         0.5     0.353554     0.0    0.000001   
506 0         0.138564   0.025         0.5     0.353554     0.0    0.000001   
    1         0.138564   0.025         0.5     0.353554     0.0    0.000001   

             tap_neutral  tap_min  tap_max  tap_step_percent  tap_pos  \
bus circuit                                                             
610 0                  0        0        0               0.0        0   
    1                  0        0        0               0.0        0   
506 0                  0        0        0               0.0        0   
    1                  0        0        0               0.0        0   

             in_service tap_side  tap_step_degree  
bus circuit                                        
610 0              True       lv              0.0  
    1              True       lv              0.0  
506 0              True       lv              0.0  
    1              True       lv              0.0

You can see that the above transformer has sn_mva = 0.025. Accordingly the active power of the load has been capped to 0.0375 (overload factor was 1.5)

In [ ]:
# Print again loads with capped values 
net.asymmetric_load.head(15)